# CLARION Model Architecture & Training: ScamRadar
### Enterprise-Grade DistilBERT Multilingual Scam Text Classifier

**Model Specification:** `distilbert-base-multilingual-cased` fine-tuned for 9-class NLP classification.\n
**Target Metric:** Multi-class classification of Indian digital fraud patterns.\n
**Training Corpus:** Curated fraud dataset with rule-based paraphrase augmentation + UCI SMS Spam Collection for legitimate class balancing.

### Target Classes:
0. `legitimate` — Verified safe interaction
1. `fake_cbi` — Fake CBI digital arrest fraud
2. `fake_ed` — Fake ED money laundering fraud
3. `customs_parcel` — Customs/drug parcel fraud
4. `court_summons` — Fake court summons fraud
5. `trai_suspension` — TRAI number suspension fraud
6. `rbi_freeze` — Fake RBI account freeze fraud
7. `narcotics_bureau` — Fake NCB narcotics fraud
8. `courier_scam` — Fake courier/delivery fraud

> ⏱️ Estimated time: ~20–30 minutes on Colab GPU (T4)

In [ ]:
# ── Phase 1: Environment Provisioning ─────────────────────────────────────────
print('Installing NLP dependencies (HuggingFace Transformers, Datasets, Accelerate)...')
!pip uninstall -y torchvision
!pip install -q transformers datasets evaluate accelerate kaggle
print('Environment provisioning complete.')

In [ ]:
# ── Phase 2: Credentials Configuration ────────────────────────────────────────
import os, json

KAGGLE_USERNAME = 'your_kaggle_username'   # ← Input required
KAGGLE_KEY      = 'your_kaggle_api_key'    # ← Input required

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
creds = {'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}
with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump(creds, f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

print(f'Kaggle authentication initialized for user: {KAGGLE_USERNAME}')

In [ ]:
# ── Phase 3: Base Data Ingestion (Legitimate Corpus) ──────────────────────────
# Utilizing the verified UCI SMS Spam Collection to represent legitimate SMS communications.
import pandas as pd, glob
from pathlib import Path

os.makedirs('/content/kaggle_sms', exist_ok=True)

print('Initiating data transfer: uciml/sms-spam-collection-dataset')
!kaggle datasets download -d uciml/sms-spam-collection-dataset -p /content/kaggle_sms --unzip

sms_files = glob.glob('/content/kaggle_sms/**/*.csv', recursive=True) + \
            glob.glob('/content/kaggle_sms/**/*.txt', recursive=True)

sms_df = None
for f in sms_files:
    try:
        sms_df = pd.read_csv(f, sep='\t', header=None, names=['label', 'text'], encoding='latin-1')
        if 'ham' in sms_df['label'].values: break
    except:
        try:
            sms_df = pd.read_csv(f, encoding='latin-1')
            sms_df.columns = ['label', 'text'] + list(sms_df.columns[2:])
            if 'ham' in sms_df['label'].values: break
        except: pass

if sms_df is not None:
    ham_df = sms_df[sms_df['label'] == 'ham'][['text']].copy()
    print(f'Successfully ingested {len(ham_df)} legitimate communication samples.')
else:
    ham_df = pd.DataFrame({'text': []})
    print('Data ingestion failed. Falling back to internal corpus.')

In [ ]:
# ── Phase 4: Threat Intelligence Corpus Construction ──────────────────────────
CURATED_DATA = [
  # ── legitimate (class 0) ──
  {'text': 'My bank sent me an OTP to confirm a transaction I made online', 'label': 0},
  {'text': 'The delivery agent called to confirm my address for the package', 'label': 0},
  {'text': 'I received a call from my credit card company about an offer', 'label': 0},
  {'text': 'My mobile operator sent an SMS about a new plan upgrade', 'label': 0},
  {'text': 'The bank called to verify a large transaction I made yesterday', 'label': 0},
  {'text': 'Got a call from insurance company for policy renewal reminder', 'label': 0},
  {'text': 'My employer deposited salary and sent me a confirmation SMS', 'label': 0},
  {'text': 'The gas agency called to schedule a cylinder delivery', 'label': 0},
  {'text': 'Received an email from IRCTC about my confirmed train ticket', 'label': 0},
  {'text': 'My friend transferred money to me and sent a payment confirmation', 'label': 0},

  # ── fake_cbi (class 1) ──
  {'text': 'A person called saying he is a CBI officer and my bank account is linked to money laundering', 'label': 1},
  {'text': 'Someone claiming to be CBI called and said I am under digital arrest for financial fraud', 'label': 1},
  {'text': 'A caller said CBI has issued a warrant for my arrest and I need to stay on video call', 'label': 1},
  {'text': 'Person said they are from CBI and my Aadhaar was found in a criminal case', 'label': 1},
  {'text': 'Got a call from someone claiming to be a CBI inspector about my bank transactions', 'label': 1},
  {'text': 'A caller threatened to arrest me via video call unless I pay to cancel the FIR', 'label': 1},
  {'text': 'Someone said I am digitally arrested by CBI and must not disconnect or speak to family', 'label': 1},
  {'text': 'CBI officer called demanding I transfer money to government account to avoid arrest', 'label': 1},
  {'text': 'Received a call claiming to be CBI with a warrant against me for money laundering case', 'label': 1},
  {'text': 'Person said my PAN card was misused and CBI is investigating me for financial crime', 'label': 1},

  # ── fake_ed (class 2) ──
  {'text': 'A caller said he is from Enforcement Directorate and my account was used for money laundering', 'label': 2},
  {'text': 'Person claiming to be ED officer called about PMLA case against my bank account', 'label': 2},
  {'text': 'Someone from ED said Rs 2 crore black money transaction was done from my account', 'label': 2},
  {'text': 'Caller said Income Tax department found undeclared foreign exchange in my name', 'label': 2},
  {'text': 'ED officer called saying my account will be seized unless I pay a settlement amount', 'label': 2},
  {'text': 'Person said Enforcement Directorate is investigating tax evasion against me', 'label': 2},
  {'text': 'Got a call claiming ED has found Rs 50 lakh suspicious transaction from my bank', 'label': 2},
  {'text': 'Someone said ED has identified me in a Rs 100 crore money laundering network', 'label': 2},
  {'text': 'Caller said to install AnyDesk for ED verification of my bank account immediately', 'label': 2},
  {'text': 'ED caller asked me to transfer money to a safe government escrow account', 'label': 2},

  # ── customs_parcel (class 3) ──
  {'text': 'Got a call saying a parcel in my name was seized at Mumbai airport with drugs inside', 'label': 3},
  {'text': 'Someone claiming to be customs officer called about an intercepted package with my Aadhaar', 'label': 3},
  {'text': 'A caller said customs found foreign currency and SIM cards in a parcel sent in my name', 'label': 3},
  {'text': 'Received call that a DHL package addressed to me contains contraband and I face arrest', 'label': 3},
  {'text': 'Person said FedEx parcel from my address was intercepted with narcotics at customs', 'label': 3},
  {'text': 'Customs officer called demanding clearance fee to release my package seized at airport', 'label': 3},
  {'text': 'Someone said I sent a courier with drugs to another country and it was seized', 'label': 3},
  {'text': 'Caller claimed customs department found illegal goods in parcel with my identity documents', 'label': 3},
  {'text': 'Got call from customs saying a package with fake currency notes has my Aadhaar attached', 'label': 3},
  {'text': 'Person claiming to be airport customs officer said I will be arrested under NDPS Act', 'label': 3},

  # ── court_summons (class 4) ──
  {'text': 'Got a call saying I missed a court hearing and a non-bailable warrant has been issued', 'label': 4},
  {'text': 'Person claiming to be a lawyer said I have a court case pending and must pay fine immediately', 'label': 4},
  {'text': 'Received call from someone saying High Court has issued summons against my name', 'label': 4},
  {'text': 'Caller said police will come to my house in 2 hours as court issued arrest warrant', 'label': 4},
  {'text': 'Someone said I am a witness in a fraud case and must appear in court or pay penalty', 'label': 4},
  {'text': 'Person claiming to be court official called demanding immediate payment to avoid contempt', 'label': 4},
  {'text': 'Got WhatsApp message with fake court notice saying I missed a hearing yesterday', 'label': 4},
  {'text': 'Caller said Supreme Court has issued notice for my financial fraud and I should pay bail', 'label': 4},
  {'text': 'Person said I need to pay court fee via UPI to avoid arrest under contempt of court', 'label': 4},
  {'text': 'Received fake court summons via WhatsApp with judge signature asking for settlement', 'label': 4},

  # ── trai_suspension (class 5) ──
  {'text': 'Got a call saying TRAI will block my mobile number in 2 hours for illegal use', 'label': 5},
  {'text': 'Caller said TRAI found my number was used to make fraudulent calls', 'label': 5},
  {'text': 'Someone claiming to be from TRAI said my SIM will be deactivated for misuse', 'label': 5},
  {'text': 'Received automated call saying press 9 to talk to TRAI officer about number suspension', 'label': 5},
  {'text': 'Person said my Jio number will be permanently blocked unless I verify Aadhaar', 'label': 5},
  {'text': 'TRAI caller said multiple spam complaints have been lodged against my number', 'label': 5},
  {'text': 'Call from TRAI saying my number was used in a cyber fraud and will be blocked', 'label': 5},
  {'text': 'Someone said they are from telecom authority and my number is under investigation', 'label': 5},
  {'text': 'Caller asked me to share OTP to complete KYC re-verification to prevent number block', 'label': 5},
  {'text': 'Person said BSNL is going to blacklist my number permanently due to regulatory violation', 'label': 5},

  # ── rbi_freeze (class 6) ──
  {'text': 'Got a call claiming to be RBI saying my account will be frozen for suspicious activity', 'label': 6},
  {'text': 'Person from Reserve Bank of India called saying my account has been flagged', 'label': 6},
  {'text': 'RBI caller said my bank account received suspicious RTGS transfers and will be blocked', 'label': 6},
  {'text': 'Someone from RBI demanded I share OTP to complete account verification', 'label': 6},
  {'text': 'Caller said my SBI account will be permanently blocked unless I verify immediately', 'label': 6},
  {'text': 'Person claiming to be RBI officer asked me to transfer money to a safe escrow account', 'label': 6},
  {'text': 'RBI person called saying my net banking will be suspended for security reasons', 'label': 6},
  {'text': 'Got call from someone claiming my bank account was used for fraudulent transactions', 'label': 6},
  {'text': 'Person said RBI investigation found my HDFC account linked to money laundering', 'label': 6},
  {'text': 'Caller asked me to install AnyDesk for RBI remote verification of my account', 'label': 6},

  # ── narcotics_bureau (class 7) ──
  {'text': 'Caller said NCB found my name in a drug trafficking network and will arrest me', 'label': 7},
  {'text': 'Person claiming to be Narcotics Control Bureau officer said I am part of drug case', 'label': 7},
  {'text': 'Someone said my name was found on a drug dealer contact list being investigated', 'label': 7},
  {'text': 'NCB caller said my bank account was used to pay drug suppliers', 'label': 7},
  {'text': 'Received call from fake NCB officer saying cocaine was sent using my Aadhaar', 'label': 7},
  {'text': 'Person said they are Anti Narcotics Cell and found my number in a drug gang database', 'label': 7},
  {'text': 'Caller claimed to be NCB inspector and asked me to pay to remove my name from case', 'label': 7},
  {'text': 'Someone said heroin consignment with my identity was seized and I face NDPS charges', 'label': 7},
  {'text': 'NCB officer called saying they have evidence of my involvement in drug supply chain', 'label': 7},
  {'text': 'Person said to keep this call secret as NCB is investigating me for narcotics', 'label': 7},

  # ── courier_scam (class 8) ──
  {'text': 'Got a call about an undelivered package I never ordered asking me to pay redelivery fee', 'label': 8},
  {'text': 'Someone claiming to be Amazon delivery agent asked for UPI payment to deliver package', 'label': 8},
  {'text': 'Received SMS saying my Flipkart order failed to deliver click link to reschedule', 'label': 8},
  {'text': 'Caller said I won a lucky prize and courier will deliver it but I need to pay customs', 'label': 8},
  {'text': 'Person from fake courier company called about package in my name needing KYC', 'label': 8},
  {'text': 'Got WhatsApp message from DTDC saying package needs payment to release from warehouse', 'label': 8},
  {'text': 'Caller said to click a phishing link to track and redeliver my pending package', 'label': 8},
  {'text': 'Someone said courier package with my address will be returned unless I pay immediately', 'label': 8},
  {'text': 'Received fake India Post SMS saying parcel needs additional fee to be delivered', 'label': 8},
  {'text': 'Caller said they are from Meesho and my order needs re-KYC before delivery', 'label': 8}
]

from collections import Counter
CLASS_NAMES = ['legitimate', 'fake_cbi', 'fake_ed', 'customs_parcel', 'court_summons',
               'trai_suspension', 'rbi_freeze', 'narcotics_bureau', 'courier_scam']
print(f'Ingested {len(CURATED_DATA)} core threat intelligence samples.')

In [ ]:
# ── Phase 5: Corpus Merging & Contextual Augmentation ─────────────────────────
import random

all_data = list(CURATED_DATA)

if len(ham_df) > 0:
    ham_sample = ham_df['text'].sample(min(300, len(ham_df)), random_state=42)
    for text in ham_sample:
        if isinstance(text, str) and len(text) > 10:
            all_data.append({'text': str(text), 'label': 0})

def contextual_paraphrase(text):
    replacements = [
        ('called', 'phoned'), ('called', 'rang me'), ('called', 'contacted me'),
        ('said', 'told me'), ('said', 'claimed'), ('said', 'stated'),
        ('my account', 'my bank account'), ('my account', 'my savings account'),
        ('person', 'individual'), ('caller', 'the person'), ('caller', 'the caller'),
        ('immediately', 'right away'), ('immediately', 'urgently'), ('immediately', 'at once'),
        ('money', 'funds'), ('pay', 'transfer'), ('pay', 'send'),
        ('arrest', 'detain'), ('arrest', 'apprehend'),
        ('someone', 'a person'), ('got a call', 'received a call'),
        ('claimed', 'pretended'), ('officer', 'official'),
    ]
    result = text
    num_replacements = random.randint(1, 3)
    for orig, replacement in random.sample(replacements, k=min(num_replacements, len(replacements))):
        if orig in result.lower():
            idx = result.lower().find(orig)
            result = result[:idx] + replacement + result[idx+len(orig):]
    return result

expanded = list(all_data)
for item in all_data:
    expanded.append({'text': contextual_paraphrase(item['text']), 'label': item['label']})
    expanded.append({'text': contextual_paraphrase(contextual_paraphrase(item['text'])), 'label': item['label']})

random.shuffle(expanded)
print(f'Corpus expansion complete. Final training set: {len(expanded)} structured sequences.')


In [ ]:
# ── Phase 6: Tokenization Pipeline ────────────────────────────────────────────
from datasets import Dataset
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME = 'distilbert-base-multilingual-cased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

hf_data = Dataset.from_list(expanded)
hf_data = hf_data.map(tokenize, batched=True, batch_size=64)
hf_data = hf_data.rename_column('label', 'labels')
from datasets import ClassLabel
hf_data = hf_data.cast_column('labels', ClassLabel(num_classes=9, names=CLASS_NAMES))
hf_data.set_format('torch')

split    = hf_data.train_test_split(test_size=0.15, seed=42, stratify_by_column='labels')
train_ds = split['train']
test_ds  = split['test']

print(f'Data stratified. Training split: {len(train_ds)} | Evaluation split: {len(test_ds)}')


In [ ]:
# ── Phase 7: Model Compilation & Training ─────────────────────────────────────
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=9,
    id2label={i: name for i, name in enumerate(CLASS_NAMES)},
    label2id={name: i for i, name in enumerate(CLASS_NAMES)},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    return {
        'accuracy':     accuracy_score(labels, predictions),
        'f1_macro':     f1_score(labels, predictions, average='macro',    zero_division=0),
        'f1_weighted':  f1_score(labels, predictions, average='weighted', zero_division=0),
    }

training_args = TrainingArguments(
    output_dir='/content/scam_distilbert',
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1, # Keep only the most recent checkpoint to save space
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=20,
    report_to='none',
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

print('Executing distributed transformer fine-tuning... (~10-15 minutes on GPU)')
trainer.train()

In [ ]:
# ── Phase 8: Performance Evaluation ───────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(test_ds)
y_pred      = np.argmax(predictions.predictions, axis=-1)
y_true      = predictions.label_ids

print('\n' + '='*60)
print('CLARION ScamRadar — Classifier Performance Report')
print('='*60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
legit_fp = cm[0][1:].sum() if len(cm) > 0 else 0
legit_total = cm[0].sum() if len(cm) > 0 else 0
fpr = legit_fp / legit_total if legit_total > 0 else 0
print(f'False Positive Rate (Legitimate classified as Scam): {fpr:.1%}')

In [ ]:
# ── Phase 9: Model Export & Deployment ────────────────────────────────────────
import shutil, os, glob

SAVE_DIR = '/content/scam_distilbert'

# 1. Force delete all heavy intermediate checkpoint folders before zipping
for checkpoint_dir in glob.glob(f'{SAVE_DIR}/checkpoint-*'):
    shutil.rmtree(checkpoint_dir, ignore_errors=True)

# 2. Save only the clean final model weights
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# 3. Zip the clean directory
ZIP_PATH = '/content/scam_distilbert.zip'
shutil.make_archive('/content/scam_distilbert', 'zip', '/content', 'scam_distilbert')

size_mb = os.path.getsize(ZIP_PATH) / (1024*1024)
print(f'✅ Model serialized and compressed: {size_mb:.0f} MB')

from google.colab import files
files.download(ZIP_PATH)

print('\nDeployment instruction:')
print('1. Extract scam_distilbert.zip')
print('2. Move the entire folder into D:\\Projects\\ET-AI\\CLARION\\backend\\saved_models\\')
print('3. Ensure the folder is strictly named `scam_distilbert` (backend expectation)')